In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set working directory
import os
project_path = '/content/drive/MyDrive/CoFT'
os.chdir(project_path)
print(f"Working directory: {os.getcwd()}")

# Install dependencies
%pip install numpy scikit-learn pandas openpyxl mne==0.20.7 mat4py einops

# Import all required libraries
import argparse
import sys
from datetime import datetime
import logging
import random
from shutil import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, cohen_kappa_score, confusion_matrix, accuracy_score

print("✅ Setup completed successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")


In [ ]:
# =============================================================================
# CONFIGURATION PARAMETERS - Modify these for your experiments
# =============================================================================

class ExperimentConfig:
    # Main experiment settings
    experiment_description = 'HAR_experiments'
    run_description = 'colab_test1'
    seed = 0
    
    # Dataset selection: 'HAR', 'sleep', 'Epilepsy', 'pFD'
    selected_dataset = 'HAR'
    data_path = 'data/'
    
    # Training mode options:
    # 'self_supervised', 'train_linear_1p', 'ft_1p', 'gen_pseudo_labels', 'SupCon', 'train_linear_SupCon_1p', 'full_run'
    training_mode = 'full_run'  # Use 'full_run' for complete pipeline
    
    # CoFT Feature Toggle
    enable_coft = True  # Set to False for baseline CA-TCC
    
    # System settings - Force GPU usage if available
    if torch.cuda.is_available():
        device = 'cuda:0'
        print(f"🚀 GPU detected: {torch.cuda.get_device_name()}")
        print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        device = 'cpu'
        print("⚠️ No GPU available, using CPU")
    
    logs_save_dir = 'experiments_logs'
    home_path = os.getcwd()

# Create configuration instance
config = ExperimentConfig()

print("📋 Experiment Configuration:")
print(f"   Dataset: {config.selected_dataset}")
print(f"   Training Mode: {config.training_mode}")
print(f"   CoFT Enabled: {config.enable_coft}")
print(f"   Device: {config.device}")
print(f"   Seed: {config.seed}")


In [ ]:
# =============================================================================
# DATASET CONFIGURATIONS
# =============================================================================

class HAR_Config(object):
    def __init__(self):
        # model configs
        self.input_channels = 9
        self.kernel_size = 8
        self.stride = 1
        self.final_out_channels = 128
        self.num_classes = 6
        self.dropout = 0.35
        self.features_len = 18
        
        # training configs
        self.num_epoch = 40
        
        # optimizer parameters
        self.beta1 = 0.9
        self.beta2 = 0.99
        self.lr = 3e-4
        
        # data parameters
        self.drop_last = True
        self.batch_size = 128
        
        self.Context_Cont = Context_Cont_configs()
        self.TC = TC_configs()
        self.augmentation = augmentations()

class sleep_Config(object):
    def __init__(self):
        # model configs
        self.input_channels = 1
        self.kernel_size = 25
        self.stride = 3
        self.final_out_channels = 128
        self.num_classes = 5
        self.dropout = 0.35
        self.features_len = 127
        
        # training configs
        self.num_epoch = 40
        
        # optimizer parameters
        self.beta1 = 0.9
        self.beta2 = 0.99
        self.lr = 3e-4
        
        # data parameters
        self.drop_last = True
        self.batch_size = 128
        
        self.Context_Cont = Context_Cont_configs()
        self.TC = TC_configs()
        self.augmentation = augmentations()

class Epilepsy_Config(object):
    def __init__(self):
        # model configs
        self.input_channels = 1
        self.kernel_size = 8
        self.stride = 1
        self.final_out_channels = 128
        self.num_classes = 2
        self.dropout = 0.35
        self.features_len = 18
        
        # training configs
        self.num_epoch = 40
        
        # optimizer parameters
        self.beta1 = 0.9
        self.beta2 = 0.99
        self.lr = 3e-4
        
        # data parameters
        self.drop_last = True
        self.batch_size = 128
        
        self.Context_Cont = Context_Cont_configs()
        self.TC = TC_configs()
        self.augmentation = augmentations()

class pFD_Config(object):
    def __init__(self):
        # model configs
        self.input_channels = 1
        self.kernel_size = 8
        self.stride = 1
        self.final_out_channels = 128
        self.num_classes = 3
        self.dropout = 0.35
        self.features_len = 18
        
        # training configs
        self.num_epoch = 40
        
        # optimizer parameters
        self.beta1 = 0.9
        self.beta2 = 0.99
        self.lr = 3e-4
        
        # data parameters
        self.drop_last = True
        self.batch_size = 128
        
        self.Context_Cont = Context_Cont_configs()
        self.TC = TC_configs()
        self.augmentation = augmentations()

# Supporting configuration classes
class augmentations(object):
    def __init__(self):
        self.jitter_scale_ratio = 1.1
        self.jitter_ratio = 0.8
        self.max_seg = 8

class Context_Cont_configs(object):
    def __init__(self):
        self.temperature = 0.2
        self.use_cosine_similarity = True

class TC_configs(object):
    def __init__(self):
        self.hidden_dim = 100
        self.timesteps = 6

# Configuration factory
def get_dataset_config(dataset_name):
    configs = {
        'HAR': HAR_Config,
        'sleep': sleep_Config,
        'Epilepsy': Epilepsy_Config,
        'pFD': pFD_Config
    }
    if dataset_name not in configs:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    return configs[dataset_name]()

print("✅ Dataset configurations loaded successfully!")


In [ ]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def set_requires_grad(model, dict_, requires_grad=True):
    for param in model.named_parameters():
        if param[0] in dict_:
            param[1].requires_grad = requires_grad

def loop_iterable(iterable):
    while True:
        yield from iterable

def fix_randomness(SEED):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

def _calc_metrics(pred_labels, true_labels, log_dir, home_path):
    pred_labels = np.array(pred_labels).astype(int)
    true_labels = np.array(true_labels).astype(int)

    # save targets
    labels_save_path = os.path.join(log_dir, "labels")
    os.makedirs(labels_save_path, exist_ok=True)
    np.save(os.path.join(labels_save_path, "predicted_labels.npy"), pred_labels)
    np.save(os.path.join(labels_save_path, "true_labels.npy"), true_labels)

    r = classification_report(true_labels, pred_labels, digits=6, output_dict=True)
    cm = confusion_matrix(true_labels, pred_labels)
    df = pd.DataFrame(r)
    df["cohen"] = cohen_kappa_score(true_labels, pred_labels)
    df["accuracy"] = accuracy_score(true_labels, pred_labels)
    df = df * 100

    # save classification report
    exp_name = os.path.split(os.path.dirname(log_dir))[-1]
    training_mode = os.path.basename(log_dir)
    file_name = f"{exp_name}_{training_mode}_classification_report.xlsx"
    report_Save_path = os.path.join(home_path, log_dir, file_name)
    df.to_excel(report_Save_path)

    # save confusion matrix
    cm_file_name = f"{exp_name}_{training_mode}_confusion_matrix.torch"
    cm_Save_path = os.path.join(home_path, log_dir, cm_file_name)
    torch.save(cm, cm_Save_path)

def _logger(logger_name, level=logging.DEBUG):
    """Method to return a custom logger with the given name and level"""
    logger = logging.getLogger(logger_name)
    
    # Clear any existing handlers to avoid duplication
    logger.handlers.clear()
    
    logger.setLevel(level)
    format_string = "%(message)s"
    log_format = logging.Formatter(format_string)
    
    # Only add file handler - avoid console duplication
    # Console output will come from print statements
    file_handler = logging.FileHandler(logger_name, mode='a')
    file_handler.setFormatter(log_format)
    logger.addHandler(file_handler)
    
    # Prevent propagation to root logger to avoid duplication
    logger.propagate = False
    
    return logger

def copy_Files(destination, data_type):
    """Copy relevant files for reproducibility"""
    destination_dir = os.path.join(destination, "model_files")
    os.makedirs(destination_dir, exist_ok=True)
    # In Colab environment, files are consolidated in notebook
    print(f"Files consolidated in notebook - saving config for {data_type}")

print("✅ Utility functions loaded successfully!")


In [ ]:
# =============================================================================
# DATA AUGMENTATIONS
# =============================================================================

def DataTransform(sample, config):
    """Apply data augmentations for self-supervised learning"""
    weak_aug = scaling(sample, config.augmentation.jitter_scale_ratio)
    strong_aug = jitter(permutation(sample, max_segments=config.augmentation.max_seg), config.augmentation.jitter_ratio)
    return weak_aug, strong_aug

def jitter(x, sigma=0.03):
    """Add simple Gaussian noise - most reliable augmentation"""
    # Convert to numpy if it's a tensor
    if hasattr(x, 'numpy'):
        x = x.numpy()
    elif not isinstance(x, np.ndarray):
        x = np.array(x)
    
    noise = np.random.normal(loc=0., scale=sigma, size=x.shape)
    return x + noise

def scaling(x, sigma=0.1):
    """Apply simple scaling augmentation"""
    # Convert to numpy if it's a tensor
    if hasattr(x, 'numpy'):
        x = x.numpy()
    elif not isinstance(x, np.ndarray):
        x = np.array(x)
    
    # Simple scaling factor per sample
    factor = np.random.normal(loc=1.0, scale=sigma, size=(x.shape[0], 1, 1))
    return x * factor

def time_shift(x, max_shift=10):
    """Simple time shifting augmentation"""
    # Convert to numpy if it's a tensor
    if hasattr(x, 'numpy'):
        x = x.numpy()
    elif not isinstance(x, np.ndarray):
        x = np.array(x)
    
    ret = np.zeros_like(x)
    for i in range(x.shape[0]):
        shift = np.random.randint(-max_shift, max_shift + 1)
        if shift > 0:
            ret[i, :, shift:] = x[i, :, :-shift]
            ret[i, :, :shift] = x[i, :, -shift:]  # Wrap around
        elif shift < 0:
            ret[i, :, :shift] = x[i, :, -shift:]
            ret[i, :, shift:] = x[i, :, :-shift]  # Wrap around
        else:
            ret[i] = x[i]
    
    return ret

def channel_shuffle(x):
    """Randomly shuffle channels"""
    # Convert to numpy if it's a tensor
    if hasattr(x, 'numpy'):
        x = x.numpy()
    elif not isinstance(x, np.ndarray):
        x = np.array(x)
    
    ret = np.zeros_like(x)
    for i in range(x.shape[0]):
        # Randomly permute channels
        channel_perm = np.random.permutation(x.shape[1])
        ret[i] = x[i, channel_perm, :]
    
    return ret

# =============================================================================
# SAMPLE DATA GENERATION (for testing when real data is not available)
# =============================================================================

def generate_sample_data(dataset_name, data_path):
    """Generate sample data for testing purposes when real data is not available."""
    
    print(f"⚠️ Generating sample data for {dataset_name} dataset...")
    
    # Get dataset configuration
    configs = get_dataset_config(dataset_name)
    
    # Sample data parameters
    if dataset_name == 'HAR':
        num_samples = 1000
        time_steps = 128
        channels = 9
        num_classes = 6
    elif dataset_name == 'sleep':
        num_samples = 800
        time_steps = 3000
        channels = 1
        num_classes = 5
    elif dataset_name == 'Epilepsy':
        num_samples = 600
        time_steps = 178
        channels = 1
        num_classes = 2
    elif dataset_name == 'pFD':
        num_samples = 400
        time_steps = 5120
        channels = 1
        num_classes = 3
    else:
        # Default values
        num_samples = 500
        time_steps = 128
        channels = configs.input_channels
        num_classes = configs.num_classes
    
    # Generate random data
    np.random.seed(42)  # For reproducibility
    
    # Create train, validation, and test splits
    train_samples = int(0.7 * num_samples)
    val_samples = int(0.15 * num_samples)
    test_samples = num_samples - train_samples - val_samples
    
    def create_dataset(n_samples):
        # Generate random time series data
        samples = np.random.randn(n_samples, channels, time_steps).astype(np.float32)
        # Generate random labels
        labels = np.random.randint(0, num_classes, n_samples)
        return {
            'samples': torch.from_numpy(samples),
            'labels': torch.from_numpy(labels)
        }
    
    # Create datasets
    train_data = create_dataset(train_samples)
    val_data = create_dataset(val_samples)
    test_data = create_dataset(test_samples)
    
    # Create 1% training data for few-shot learning
    train_1p_samples = max(1, int(0.01 * train_samples))
    train_1p_data = create_dataset(train_1p_samples)
    
    # Save the datasets
    os.makedirs(data_path, exist_ok=True)
    torch.save(train_data, os.path.join(data_path, "train.pt"))
    torch.save(val_data, os.path.join(data_path, "val.pt"))
    torch.save(test_data, os.path.join(data_path, "test.pt"))
    torch.save(train_1p_data, os.path.join(data_path, "train_1perc.pt"))
    
    print(f"✅ Generated sample data:")
    print(f"   Train: {train_samples} samples, shape: {train_data['samples'].shape}")
    print(f"   Val: {val_samples} samples, shape: {val_data['samples'].shape}")
    print(f"   Test: {test_samples} samples, shape: {test_data['samples'].shape}")
    print(f"   Train 1%: {train_1p_samples} samples, shape: {train_1p_data['samples'].shape}")

# =============================================================================
# DATA LOADING
# =============================================================================

class Load_Dataset(Dataset):
    def __init__(self, dataset, config, training_mode):
        self.training_mode = training_mode
        self.config = config
        X_train = dataset["samples"]
        y_train = dataset["labels"]

        if len(X_train.shape) < 3:
            X_train = X_train.unsqueeze(2)

        if X_train.shape.index(min(X_train.shape)) != 1:  # make sure the Channels in second dim
            X_train = X_train.permute(0, 2, 1)

        if isinstance(X_train, np.ndarray):
            self.x_data = torch.from_numpy(X_train)
            self.y_data = torch.from_numpy(y_train).long()
        else:
            self.x_data = X_train
            self.y_data = y_train

        # Keep data on CPU for DataLoader
        self.x_data = self.x_data.cpu().float()
        self.y_data = self.y_data.cpu()

        self.len = X_train.shape[0]
        
        print(f"✅ Dataset initialized: {self.len} samples, shape: {self.x_data.shape}")

    def apply_augmentations(self, x):
        """Apply augmentations to a single sample on-the-fly"""
        try:
            # Convert to numpy for augmentation
            x_np = x.numpy() if hasattr(x, 'numpy') else x
            
            # Add batch dimension if needed (single sample: channels x time -> 1 x channels x time)
            if len(x_np.shape) == 2:
                x_np = x_np[np.newaxis, ...]  # Add batch dimension
            
            # Apply weak augmentation: scaling + light jitter
            weak_aug = scaling(x_np.copy(), sigma=0.05)
            weak_aug = jitter(weak_aug, sigma=0.02)
            
            # Apply strong augmentation: time shift + channel shuffle + jitter
            strong_aug = time_shift(x_np.copy(), max_shift=8)
            strong_aug = channel_shuffle(strong_aug)
            strong_aug = jitter(strong_aug, sigma=0.05)
            
            # Remove batch dimension and convert back to tensors
            weak_aug = torch.from_numpy(weak_aug.squeeze(0)).float()  # Remove batch dim
            strong_aug = torch.from_numpy(strong_aug.squeeze(0)).float()  # Remove batch dim
            
            return weak_aug, strong_aug
            
        except Exception as e:
            print(f"Warning: Augmentation failed for sample, using fallback: {e}")
            # Fallback: return slightly modified versions with simple noise
            noise1 = torch.randn_like(x) * 0.01
            noise2 = torch.randn_like(x) * 0.02
            return x + noise1, x + noise2

    def __getitem__(self, index):
        x = self.x_data[index]
        y = self.y_data[index]
        
        if self.training_mode == "self_supervised" or self.training_mode == "SupCon":
            # Apply augmentations on-the-fly
            aug1, aug2 = self.apply_augmentations(x)
            return x, y, aug1, aug2
        else:
            return x, y, x, x

    def __len__(self):
        return self.len

def data_generator(data_path, configs, training_mode):
    batch_size = configs.batch_size
    
    # Optimized DataLoader configuration based on device
    if torch.cuda.is_available():
        num_workers = 4  # More workers for GPU
        pin_memory = True  # Enable for faster GPU transfer
        persistent_workers = True  # Keep workers alive for efficiency
        print("🚀 Using GPU-optimized DataLoader settings")
    else:
        num_workers = 2  # Conservative setting for CPU
        pin_memory = False  # Not needed for CPU
        persistent_workers = False  # Avoid threading issues on CPU
        print("⚙️ Using CPU-optimized DataLoader settings")

    # Debug: Check if data path exists
    print(f"Looking for data in: {data_path}")
    
    # Extract dataset name from path for sample data generation
    dataset_name = os.path.basename(data_path)
    
    # Determine which training file to load
    if "_1p" in training_mode:
        train_file = "train_1perc.pt"
    elif "_5p" in training_mode:
        train_file = "train_5perc.pt"
    elif "_10p" in training_mode:
        train_file = "train_10perc.pt"
    elif "_50p" in training_mode:
        train_file = "train_50perc.pt"
    elif "_75p" in training_mode:
        train_file = "train_75perc.pt"
    elif training_mode == "SupCon":
        train_file = "pseudo_train_data.pt"
    else:
        train_file = "train.pt"

    # Check if files exist
    train_path = os.path.join(data_path, train_file)
    val_path = os.path.join(data_path, "val.pt")
    test_path = os.path.join(data_path, "test.pt")
    
    print(f"Checking files:")
    print(f"  Train: {train_path} - {'EXISTS' if os.path.exists(train_path) else 'MISSING'}")
    print(f"  Val:   {val_path} - {'EXISTS' if os.path.exists(val_path) else 'MISSING'}")
    print(f"  Test:  {test_path} - {'EXISTS' if os.path.exists(test_path) else 'MISSING'}")

    # Generate sample data if real data doesn't exist
    if not (os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path)):
        print(f"📋 Real data files not found. Generating sample data for testing...")
        generate_sample_data(dataset_name, data_path)
        print(f"✅ Sample data generated successfully!")

    # Load datasets with error handling
    try:
        train_dataset = torch.load(train_path)
        print(f"✅ Loaded train dataset: {type(train_dataset)}")
        if isinstance(train_dataset, dict):
            print(f"   Keys: {list(train_dataset.keys())}")
            if 'samples' in train_dataset:
                print(f"   Sample shape: {train_dataset['samples'].shape}")
            if 'labels' in train_dataset:
                print(f"   Labels shape: {train_dataset['labels'].shape}")
    except Exception as e:
        raise RuntimeError(f"Failed to load {train_path}: {e}")
    
    try:
        valid_dataset = torch.load(val_path)
        print(f"✅ Loaded validation dataset")
    except Exception as e:
        raise RuntimeError(f"Failed to load {val_path}: {e}")
    
    try:
        test_dataset = torch.load(test_path)
        print(f"✅ Loaded test dataset")
    except Exception as e:
        raise RuntimeError(f"Failed to load {test_path}: {e}")

    train_dataset = Load_Dataset(train_dataset, configs, training_mode)
    valid_dataset = Load_Dataset(valid_dataset, configs, training_mode)
    test_dataset = Load_Dataset(test_dataset, configs, training_mode)

    if train_dataset.__len__() < batch_size:
        batch_size = 16

    train_loader = torch.utils.data.DataLoader(
        dataset=train_dataset, 
        batch_size=batch_size,
        shuffle=True, 
        drop_last=configs.drop_last, 
        num_workers=num_workers, 
        pin_memory=pin_memory,
        persistent_workers=persistent_workers if num_workers > 0 else False
    )
    
    valid_loader = torch.utils.data.DataLoader(
        dataset=valid_dataset, 
        batch_size=batch_size,
        shuffle=False, 
        drop_last=configs.drop_last, 
        num_workers=num_workers, 
        pin_memory=pin_memory,
        persistent_workers=persistent_workers if num_workers > 0 else False
    )

    test_loader = torch.utils.data.DataLoader(
        dataset=test_dataset, 
        batch_size=batch_size,
        shuffle=False, 
        drop_last=False, 
        num_workers=num_workers, 
        pin_memory=pin_memory,
        persistent_workers=persistent_workers if num_workers > 0 else False
    )
    
    return train_loader, valid_loader, test_loader

print("✅ Data loading and augmentations loaded successfully!")


In [ ]:
# =============================================================================
# ATTENTION MECHANISM FOR TRANSFORMER
# =============================================================================
from einops import rearrange, repeat
import math

class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.head_dim = dim // heads

        self.to_qkv = nn.Linear(dim, dim * 3, bias=False)
        self.to_out = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x, mask=None):
        b, n, _, h = *x.shape, self.heads
        
        # Efficient QKV computation
        qkv = self.to_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        
        # Reshape for multi-head attention  
        q = q.view(b, n, h, self.head_dim).transpose(1, 2)
        k = k.view(b, n, h, self.head_dim).transpose(1, 2)
        v = v.view(b, n, h, self.head_dim).transpose(1, 2)

        # Use matmul instead of einsum for better performance
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if mask is not None:
            mask = F.pad(mask.flatten(1), (1, 0), value=True)
            assert mask.shape[-1] == attn_scores.shape[-1], 'mask has incorrect dimensions'
            mask = mask[:, None, :] * mask[:, :, None]
            attn_scores.masked_fill_(~mask, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        out = torch.matmul(attn_weights, v)
        out = out.transpose(1, 2).contiguous().view(b, n, -1)

        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, mlp_dim, dropout):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Residual(PreNorm(dim, Attention(dim, heads=heads, dropout=dropout))),
                Residual(PreNorm(dim, FeedForward(dim, mlp_dim, dropout=dropout)))
            ]))

    def forward(self, x, mask=None):
        for attn, ff in self.layers:
            x = attn(x, mask=mask)
            x = ff(x)
        return x

class Seq_Transformer(nn.Module):
    def __init__(self, *, patch_size, dim, depth, heads, mlp_dim, channels=1, dropout=0.1):
        super().__init__()
        patch_dim = channels * patch_size
        self.patch_to_embedding = nn.Linear(patch_dim, dim)
        self.c_token = nn.Parameter(torch.randn(1, 1, dim))
        self.transformer = Transformer(dim, depth, heads, mlp_dim, dropout)
        self.to_c_token = nn.Identity()

    def forward(self, forward_seq):
        x = self.patch_to_embedding(forward_seq)
        b, n, _ = x.shape
        c_tokens = repeat(self.c_token, '() n d -> b n d', b=b)
        x = torch.cat((c_tokens, x), dim=1)
        x = self.transformer(x)
        c_t = self.to_c_token(x[:, 0])
        return c_t

print("✅ Attention mechanisms loaded successfully!")


In [ ]:
# =============================================================================
# CORE MODEL ARCHITECTURES
# =============================================================================

class base_Model(nn.Module):
    def __init__(self, configs):
        super(base_Model, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv1d(configs.input_channels, 32, kernel_size=configs.kernel_size,
                      stride=configs.stride, bias=False, padding=(configs.kernel_size // 2)),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=1),
            nn.Dropout(configs.dropout)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=8, stride=1, bias=False, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=1)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv1d(64, configs.final_out_channels, kernel_size=8, stride=1, bias=False, padding=4),
            nn.BatchNorm1d(configs.final_out_channels),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=1),
        )

        model_output_dim = configs.features_len
        self.logits = nn.Linear(model_output_dim * configs.final_out_channels, configs.num_classes)

    def forward(self, x_in):
        x = self.conv_block1(x_in)
        x = self.conv_block2(x)
        x = self.conv_block3(x)

        x_flat = x.reshape(x.shape[0], -1)
        logits = self.logits(x_flat)
        return logits, x

class TC(nn.Module):
    def __init__(self, configs, device):
        super(TC, self).__init__()
        self.num_channels = configs.final_out_channels
        self.timestep = configs.TC.timesteps
        self.Wk = nn.ModuleList([nn.Linear(configs.TC.hidden_dim, self.num_channels) for i in range(self.timestep)])
        self.lsoftmax = nn.LogSoftmax(dim=-1)
        self.device = device

        self.projection_head = nn.Sequential(
            nn.Linear(configs.TC.hidden_dim, configs.final_out_channels // 2),
            nn.BatchNorm1d(configs.final_out_channels // 2),
            nn.ReLU(inplace=True),
            nn.Linear(configs.final_out_channels // 2, configs.final_out_channels // 4),
        )

        self.seq_transformer = Seq_Transformer(patch_size=self.num_channels, dim=configs.TC.hidden_dim, depth=4,
                                               heads=4, mlp_dim=64)

    def forward(self, z_aug1, z_aug2):
        seq_len = z_aug1.shape[2]

        z_aug1 = z_aug1.transpose(1, 2)
        z_aug2 = z_aug2.transpose(1, 2)

        batch = z_aug1.shape[0]
        t_samples = torch.randint(seq_len - self.timestep, size=(1,)).long().to(
            self.device)  # randomly pick time stamps

        nce = 0  # average over timestep and batch
        encode_samples = torch.empty((self.timestep, batch, self.num_channels)).float().to(self.device)

        for i in np.arange(1, self.timestep + 1):
            encode_samples[i - 1] = z_aug2[:, t_samples + i, :].view(batch, self.num_channels)
        forward_seq = z_aug1[:, :t_samples + 1, :]

        c_t = self.seq_transformer(forward_seq)

        pred = torch.empty((self.timestep, batch, self.num_channels)).float().to(self.device)
        for i in np.arange(0, self.timestep):
            linear = self.Wk[i]
            pred[i] = linear(c_t)
        for i in np.arange(0, self.timestep):
            total = torch.mm(encode_samples[i], torch.transpose(pred[i], 0, 1))
            nce += torch.sum(torch.diag(self.lsoftmax(total)))
        nce /= -1. * batch * self.timestep
        return nce, self.projection_head(c_t)

# =============================================================================
# COFT FREQUENCY MODEL
# =============================================================================

class FrequencyModel(nn.Module):
    """Frequency-domain branch for CoFT architecture."""
    
    def __init__(self, configs):
        super(FrequencyModel, self).__init__()
        
        self.input_channels = configs.input_channels
        self.fft_norm = 'ortho'  # Orthogonal normalization for stable gradients
        
        # Frequency-domain convolution blocks
        self.freq_conv_block1 = nn.Sequential(
            nn.Conv1d(configs.input_channels * 2, 32, kernel_size=configs.kernel_size,  # *2 for real/imag
                      stride=configs.stride, bias=False, padding=(configs.kernel_size // 2)),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=1),
            nn.Dropout(configs.dropout)
        )
        
        self.freq_conv_block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=8, stride=1, bias=False, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=1)
        )
        
        self.freq_conv_block3 = nn.Sequential(
            nn.Conv1d(64, configs.final_out_channels, kernel_size=8, stride=1, bias=False, padding=4),
            nn.BatchNorm1d(configs.final_out_channels),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=1),
        )
        
        # Frequency-specific classifier
        self.num_classes = configs.num_classes
        self.freq_logits = None  # Will be initialized in first forward pass
        
    def forward(self, x_in):
        # Ensure input is 3D for FFT processing
        if len(x_in.shape) == 4:
            x_in = x_in.squeeze(2)
        
        # Apply FFT to convert to frequency domain
        x_fft = torch.fft.rfft(x_in, norm=self.fft_norm)  # Real FFT for real-valued input
        
        # Convert complex to real representation (magnitude and phase)
        magnitude = torch.abs(x_fft)
        phase = torch.angle(x_fft)
        
        # Concatenate magnitude and phase as separate channels
        x_freq = torch.cat([magnitude, phase], dim=1)  # [batch, channels*2, freq_bins]
        
        # Process through frequency-specific conv blocks
        x = self.freq_conv_block1(x_freq)
        x = self.freq_conv_block2(x)
        x = self.freq_conv_block3(x)
        
        # Flatten for classification - calculate the actual dimensions dynamically
        x_flat = x.reshape(x.shape[0], -1)
        
        # Initialize linear layer if this is the first forward pass
        if self.freq_logits is None:
            actual_features = x_flat.shape[1]
            self.freq_logits = nn.Linear(actual_features, self.num_classes).to(x_flat.device)
        
        freq_logits = self.freq_logits(x_flat)
        
        return freq_logits, x

class FrequencyContrastive(nn.Module):
    """Frequency-domain contrastive learning module."""
    
    def __init__(self, configs, device):
        super(FrequencyContrastive, self).__init__()
        self.device = device
        
        # Projection head for frequency contrastive learning
        self.projection_head = nn.Sequential(
            nn.Linear(configs.final_out_channels, configs.final_out_channels // 2),
            nn.BatchNorm1d(configs.final_out_channels // 2),
            nn.ReLU(inplace=True),
            nn.Linear(configs.final_out_channels // 2, configs.final_out_channels // 4),
        )
        
    def forward(self, freq_features1, freq_features2):
        # Average pooling to get sequence-level features
        feat1 = F.adaptive_avg_pool1d(freq_features1, 1).squeeze(-1)
        feat2 = F.adaptive_avg_pool1d(freq_features2, 1).squeeze(-1)
        
        # Project to contrastive space
        proj1 = self.projection_head(feat1)
        proj2 = self.projection_head(feat2)
        
        return 0.0, torch.cat([proj1, proj2], dim=0)  # Return 0 loss, combined projections

print("✅ Core model architectures loaded successfully!")


In [ ]:
# =============================================================================
# LOSS FUNCTIONS
# =============================================================================

class NTXentLoss(torch.nn.Module):
    def __init__(self, device, batch_size, temperature, use_cosine_similarity):
        super(NTXentLoss, self).__init__()
        self.batch_size = batch_size
        self.temperature = temperature
        self.device = device
        self.softmax = torch.nn.Softmax(dim=-1)
        self.mask_samples_from_same_repr = self._get_correlated_mask().type(torch.bool)
        self.similarity_function = self._get_similarity_function(use_cosine_similarity)
        self.criterion = torch.nn.CrossEntropyLoss(reduction="sum")

    def _get_similarity_function(self, use_cosine_similarity):
        if use_cosine_similarity:
            self._cosine_similarity = torch.nn.CosineSimilarity(dim=-1)
            return self._cosine_simililarity
        else:
            return self._dot_simililarity

    def _get_correlated_mask(self):
        diag = np.eye(2 * self.batch_size)
        l1 = np.eye((2 * self.batch_size), 2 * self.batch_size, k=-self.batch_size)
        l2 = np.eye((2 * self.batch_size), 2 * self.batch_size, k=self.batch_size)
        mask = torch.from_numpy((diag + l1 + l2))
        mask = (1 - mask).type(torch.bool)
        return mask.to(self.device)

    @staticmethod
    def _dot_simililarity(x, y):
        v = torch.tensordot(x.unsqueeze(1), y.T.unsqueeze(0), dims=2)
        return v

    def _cosine_simililarity(self, x, y):
        v = self._cosine_similarity(x.unsqueeze(1), y.unsqueeze(0))
        return v

    def forward(self, zis, zjs):
        representations = torch.cat([zjs, zis], dim=0)
        similarity_matrix = self.similarity_function(representations, representations)

        # filter out the scores from the positive samples
        l_pos = torch.diag(similarity_matrix, self.batch_size)
        r_pos = torch.diag(similarity_matrix, -self.batch_size)
        positives = torch.cat([l_pos, r_pos]).view(2 * self.batch_size, 1)

        negatives = similarity_matrix[self.mask_samples_from_same_repr].view(2 * self.batch_size, -1)

        logits = torch.cat((positives, negatives), dim=1)
        logits /= self.temperature

        labels = torch.zeros(2 * self.batch_size).to(self.device).long()
        loss = self.criterion(logits, labels)

        return loss / (2 * self.batch_size)

class SupConLoss(torch.nn.Module):
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf."""

    def __init__(self, device, temperature=0.2, contrast_mode='all'):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.device = device

    def forward(self, features, labels=None, mask=None):
        device = self.device

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...], at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)

        # loss
        loss = - self.temperature * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

print("✅ Loss functions loaded successfully!")


In [ ]:
# =============================================================================
# TRAINING FUNCTIONS
# =============================================================================

def Trainer(model, temporal_contr_model, model_optimizer, temp_cont_optimizer, train_dl, valid_dl, test_dl, device,
            logger, config, experiment_log_dir, training_mode):
    """Original CA-TCC trainer"""
    logger.debug("Training started ....")

    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(model_optimizer, 'min')

    for epoch in range(1, config.num_epoch + 1):
        # Train and validate
        train_loss, train_acc = model_train(model, temporal_contr_model, model_optimizer, temp_cont_optimizer,
                                            criterion, train_dl, config, device, training_mode)
        valid_loss, valid_acc, _, _ = model_evaluate(model, temporal_contr_model, valid_dl, device, training_mode)
        if (training_mode != "self_supervised") and (training_mode != "SupCon"):
            scheduler.step(valid_loss)

        logger.debug(f'\nEpoch : {epoch}\n'
                     f'Train Loss     : {train_loss:2.4f}\t | \tTrain Accuracy     : {train_acc:2.4f}\n'
                     f'Valid Loss     : {valid_loss:2.4f}\t | \tValid Accuracy     : {valid_acc:2.4f}')

    # save the model after training
    os.makedirs(os.path.join(experiment_log_dir, "saved_models"), exist_ok=True)
    chkpoint = {'model_state_dict': model.state_dict(),
                'temporal_contr_model_state_dict': temporal_contr_model.state_dict()}
    torch.save(chkpoint, os.path.join(experiment_log_dir, "saved_models", f'ckp_last.pt'))

    if (training_mode != "self_supervised") and (training_mode != "SupCon"):
        # evaluate on the test set
        logger.debug('\nEvaluate on the Test set:')
        test_loss, test_acc, _, _ = model_evaluate(model, temporal_contr_model, test_dl, device, training_mode)
        logger.debug(f'Test loss      :{test_loss:2.4f}\t | Test Accuracy      : {test_acc:2.4f}')

    logger.debug("\n################## Training is Done! #########################")

def CoFTTrainer(model, temporal_contr_model, frequency_model, frequency_contr_model,
                model_optimizer, temporal_contr_optimizer, frequency_optimizer, frequency_contr_optimizer,
                train_dl, valid_dl, test_dl, device, logger, config, experiment_log_dir, training_mode, enable_coft):
    """Enhanced trainer with CoFT functionality"""
    logger.debug("CoFT Training started ....")

    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(model_optimizer, 'min')

    for epoch in range(1, config.num_epoch + 1):
        # Train and validate with CoFT
        train_loss, train_acc = coft_model_train(model, temporal_contr_model, frequency_model, frequency_contr_model,
                                                model_optimizer, temporal_contr_optimizer, frequency_optimizer,
                                                frequency_contr_optimizer, criterion, train_dl, config, device, training_mode)
        
        valid_loss, valid_acc, _, _ = coft_model_evaluate(model, temporal_contr_model, frequency_model, 
                                                         valid_dl, device, training_mode)
        
        if (training_mode != "self_supervised") and (training_mode != "SupCon"):
            scheduler.step(valid_loss)

        logger.debug(f'\nEpoch : {epoch}\n'
                     f'Train Loss     : {train_loss:2.4f}\t | \tTrain Accuracy     : {train_acc:2.4f}\n'
                     f'Valid Loss     : {valid_loss:2.4f}\t | \tValid Accuracy     : {valid_acc:2.4f}')

    # save the model after training
    os.makedirs(os.path.join(experiment_log_dir, "saved_models"), exist_ok=True)
    chkpoint = {
        'model_state_dict': model.state_dict(),
        'temporal_contr_model_state_dict': temporal_contr_model.state_dict(),
        'frequency_model_state_dict': frequency_model.state_dict() if frequency_model else None,
        'frequency_contr_model_state_dict': frequency_contr_model.state_dict() if frequency_contr_model else None
    }
    torch.save(chkpoint, os.path.join(experiment_log_dir, "saved_models", f'ckp_last.pt'))

    if (training_mode != "self_supervised") and (training_mode != "SupCon"):
        # evaluate on the test set with CoFT
        logger.debug('\nEvaluate on the Test set:')
        test_loss, test_acc, _, _ = coft_model_evaluate(model, temporal_contr_model, frequency_model, 
                                                       test_dl, device, training_mode)
        logger.debug(f'Test loss      :{test_loss:2.4f}\t | Test Accuracy      : {test_acc:2.4f}')

    logger.debug("\n################## CoFT Training is Done! #########################")

print("✅ Trainer functions loaded successfully!")


In [ ]:
# =============================================================================
# CORE TRAINING & EVALUATION FUNCTIONS
# =============================================================================

def model_train(model, temporal_contr_model, model_optimizer, temp_cont_optimizer, criterion, train_loader, config,
                device, training_mode):
    """Original training function"""
    total_loss = []
    total_acc = []
    model.train()
    temporal_contr_model.train()

    for batch_idx, (data, labels, aug1, aug2) in enumerate(train_loader):
        # send to device
        data, labels = data.float().to(device), labels.long().to(device)
        aug1, aug2 = aug1.float().to(device), aug2.float().to(device)

        # optimizer
        model_optimizer.zero_grad()
        temp_cont_optimizer.zero_grad()

        if training_mode == "self_supervised" or training_mode == "SupCon":
            predictions1, features1 = model(aug1)
            predictions2, features2 = model(aug2)

            # normalize projection feature vectors
            features1 = F.normalize(features1, dim=1)
            features2 = F.normalize(features2, dim=1)

            temp_cont_loss1, temp_cont_feat1 = temporal_contr_model(features1, features2)
            temp_cont_loss2, temp_cont_feat2 = temporal_contr_model(features2, features1)

        if training_mode == "self_supervised":
            lambda1 = 1
            lambda2 = 0.7
            nt_xent_criterion = NTXentLoss(device, config.batch_size, config.Context_Cont.temperature,
                                           config.Context_Cont.use_cosine_similarity)
            loss = (temp_cont_loss1 + temp_cont_loss2) * lambda1 + nt_xent_criterion(temp_cont_feat1, temp_cont_feat2) * lambda2

        elif training_mode == "SupCon":
            lambda1 = 0.01
            lambda2 = 0.1
            Sup_contrastive_criterion = SupConLoss(device)

            supCon_features = torch.cat([temp_cont_feat1.unsqueeze(1), temp_cont_feat2.unsqueeze(1)], dim=1)
            loss = (temp_cont_loss1 + temp_cont_loss2) * lambda1 + Sup_contrastive_criterion(supCon_features,
                                                                                             labels) * lambda2

        else:
            output = model(data)
            predictions, features = output
            loss = criterion(predictions, labels)
            total_acc.append(labels.eq(predictions.detach().argmax(dim=1)).float().mean())

        total_loss.append(loss.item())

        loss.backward()
        model_optimizer.step()
        if training_mode == "self_supervised" or training_mode == "SupCon":
            temp_cont_optimizer.step()

    total_loss = torch.tensor(total_loss).mean()

    if (training_mode == "self_supervised") or (training_mode == "SupCon"):
        total_acc = 0
    else:
        total_acc = torch.tensor(total_acc).mean()
    return total_loss, total_acc

def model_evaluate(model, temporal_contr_model, test_dl, device, training_mode):
    """Original evaluation function"""
    model.eval()
    temporal_contr_model.eval()

    total_loss = []
    total_acc = []

    criterion = nn.CrossEntropyLoss()
    outs = np.array([])
    trgs = np.array([])

    with torch.no_grad():
        for data, labels, _, _ in test_dl:
            data, labels = data.float().to(device), labels.long().to(device)

            if (training_mode == "self_supervised") or (training_mode == "SupCon"):
                pass
            else:
                output = model(data)

            # compute loss
            if (training_mode != "self_supervised") and (training_mode != "SupCon"):
                predictions, features = output
                loss = criterion(predictions, labels)
                total_acc.append(labels.eq(predictions.detach().argmax(dim=1)).float().mean())
                total_loss.append(loss.item())

                pred = predictions.max(1, keepdim=True)[1]  # get the index of the max log-probability
                outs = np.append(outs, pred.cpu().numpy())
                trgs = np.append(trgs, labels.data.cpu().numpy())

    if (training_mode == "self_supervised") or (training_mode == "SupCon"):
        total_loss = 0
        total_acc = 0
        return total_loss, total_acc, [], []
    else:
        total_loss = torch.tensor(total_loss).mean()  # average loss
        total_acc = torch.tensor(total_acc).mean()  # average acc
        return total_loss, total_acc, outs, trgs

def gen_pseudo_labels(model, dataloader, device, experiment_log_dir):
    """Generate pseudo labels for SupCon training"""
    model.eval()
    softmax = nn.Softmax(dim=1)

    # saving output data
    all_pseudo_labels = np.array([])
    all_labels = np.array([])
    all_data = []

    with torch.no_grad():
        for data, labels, _, _ in dataloader:
            data = data.float().to(device)
            labels = labels.view((-1)).long().to(device)

            # forward pass
            predictions, features = model(data)

            normalized_preds = softmax(predictions)
            pseudo_labels = normalized_preds.max(1, keepdim=True)[1].squeeze()
            all_pseudo_labels = np.append(all_pseudo_labels, pseudo_labels.cpu().numpy())

            all_labels = np.append(all_labels, labels.cpu().numpy())
            all_data.append(data)

    all_data = torch.cat(all_data, dim=0)

    data_save = dict()
    data_save["samples"] = all_data.cpu()
    data_save["labels"] = torch.LongTensor(torch.from_numpy(all_pseudo_labels).long())
    file_name = f"pseudo_train_data.pt"
    torch.save(data_save, os.path.join(experiment_log_dir, file_name))
    print("Pseudo labels generated ...")

print("✅ Core training functions loaded successfully!")


In [ ]:
# =============================================================================
# COFT TRAINING & EVALUATION FUNCTIONS
# =============================================================================

def coft_model_train(model, temporal_contr_model, frequency_model, frequency_contr_model,
                     model_optimizer, temporal_contr_optimizer, frequency_optimizer, frequency_contr_optimizer,
                     criterion, train_loader, config, device, training_mode):
    """CoFT enhanced training function"""
    total_loss = []
    total_acc = []
    model.train()
    temporal_contr_model.train()
    if frequency_model:
        frequency_model.train()
    if frequency_contr_model:
        frequency_contr_model.train()

    for batch_idx, (data, labels, aug1, aug2) in enumerate(train_loader):
        # send to device
        data, labels = data.float().to(device), labels.long().to(device)
        aug1, aug2 = aug1.float().to(device), aug2.float().to(device)

        # optimizer
        model_optimizer.zero_grad()
        temporal_contr_optimizer.zero_grad()
        if frequency_optimizer:
            frequency_optimizer.zero_grad()
        if frequency_contr_optimizer:
            frequency_contr_optimizer.zero_grad()

        if training_mode == "self_supervised" or training_mode == "SupCon":
            # Temporal branch
            predictions1, features1 = model(aug1)
            predictions2, features2 = model(aug2)

            # normalize projection feature vectors
            features1 = F.normalize(features1, dim=1)
            features2 = F.normalize(features2, dim=1)

            temp_cont_loss1, temp_cont_feat1 = temporal_contr_model(features1, features2)
            temp_cont_loss2, temp_cont_feat2 = temporal_contr_model(features2, features1)

            # Frequency branch (CoFT)
            freq_loss = 0
            if frequency_model and frequency_contr_model:
                freq_pred1, freq_feat1 = frequency_model(aug1)
                freq_pred2, freq_feat2 = frequency_model(aug2)
                
                freq_cont_loss, freq_cont_feat = frequency_contr_model(freq_feat1, freq_feat2)
                freq_loss = freq_cont_loss * 0.1  # Frequency contrastive weight

        if training_mode == "self_supervised":
            lambda1 = 1
            lambda2 = 0.7
            nt_xent_criterion = NTXentLoss(device, config.batch_size, config.Context_Cont.temperature,
                                           config.Context_Cont.use_cosine_similarity)
            
            temporal_loss = (temp_cont_loss1 + temp_cont_loss2) * lambda1 + nt_xent_criterion(temp_cont_feat1, temp_cont_feat2) * lambda2
            loss = temporal_loss + freq_loss

        elif training_mode == "SupCon":
            lambda1 = 0.01
            lambda2 = 0.1
            Sup_contrastive_criterion = SupConLoss(device)

            supCon_features = torch.cat([temp_cont_feat1.unsqueeze(1), temp_cont_feat2.unsqueeze(1)], dim=1)
            temporal_loss = (temp_cont_loss1 + temp_cont_loss2) * lambda1 + Sup_contrastive_criterion(supCon_features, labels) * lambda2
            loss = temporal_loss + freq_loss

        else:
            # Supervised training
            output = model(data)
            predictions, features = output
            temporal_loss = criterion(predictions, labels)
            
            # Add frequency branch predictions
            freq_loss = 0
            if frequency_model:
                freq_predictions, _ = frequency_model(data)
                freq_loss = criterion(freq_predictions, labels) * 0.1  # Co-training weight
            
            loss = temporal_loss + freq_loss
            total_acc.append(labels.eq(predictions.detach().argmax(dim=1)).float().mean())

        total_loss.append(loss.item())

        loss.backward()
        model_optimizer.step()
        temporal_contr_optimizer.step()
        if frequency_optimizer:
            frequency_optimizer.step()
        if frequency_contr_optimizer:
            frequency_contr_optimizer.step()

    total_loss = torch.tensor(total_loss).mean()

    if (training_mode == "self_supervised") or (training_mode == "SupCon"):
        total_acc = 0
    else:
        total_acc = torch.tensor(total_acc).mean()
    return total_loss, total_acc

def coft_model_evaluate(model, temporal_contr_model, frequency_model, test_dl, device, training_mode):
    """CoFT enhanced evaluation function"""
    model.eval()
    temporal_contr_model.eval()
    if frequency_model:
        frequency_model.eval()

    total_loss = []
    total_acc = []

    criterion = nn.CrossEntropyLoss()
    outs = np.array([])
    trgs = np.array([])

    with torch.no_grad():
        for data, labels, _, _ in test_dl:
            data, labels = data.float().to(device), labels.long().to(device)

            if (training_mode == "self_supervised") or (training_mode == "SupCon"):
                pass
            else:
                output = model(data)
                predictions, features = output
                
                # Ensemble with frequency predictions if available
                if frequency_model:
                    freq_predictions, _ = frequency_model(data)
                    # Simple ensemble (average)
                    ensemble_predictions = (predictions + freq_predictions) / 2
                    loss = criterion(ensemble_predictions, labels)
                    total_acc.append(labels.eq(ensemble_predictions.detach().argmax(dim=1)).float().mean())
                    pred = ensemble_predictions.max(1, keepdim=True)[1]
                else:
                    loss = criterion(predictions, labels)
                    total_acc.append(labels.eq(predictions.detach().argmax(dim=1)).float().mean())
                    pred = predictions.max(1, keepdim=True)[1]
                
                total_loss.append(loss.item())
                outs = np.append(outs, pred.cpu().numpy())
                trgs = np.append(trgs, labels.data.cpu().numpy())

    if (training_mode == "self_supervised") or (training_mode == "SupCon"):
        total_loss = 0
        total_acc = 0
        return total_loss, total_acc, [], []
    else:
        total_loss = torch.tensor(total_loss).mean()  # average loss
        total_acc = torch.tensor(total_acc).mean()  # average acc
        return total_loss, total_acc, outs, trgs

print("✅ CoFT training functions loaded successfully!")


In [ ]:
# =============================================================================
# MAIN EXECUTION FUNCTION
# =============================================================================

def execute_training_mode(args, mode_name, overall_start_time):
    """Execute a single training mode with proper setup and cleanup."""
    print(f"\n{'='*60}")
    print(f"🚀 Starting Training Mode: {mode_name}")
    print(f"{'='*60}")
    
    # Update args for current mode
    args.training_mode = mode_name
    training_mode = mode_name
    
    device = torch.device(args.device)
    experiment_description = args.experiment_description
    data_type = args.selected_dataset.replace("-", "_")
    run_description = args.run_description

    logs_save_dir = args.logs_save_dir
    os.makedirs(logs_save_dir, exist_ok=True)

    # Get dataset configuration
    configs = get_dataset_config(data_type)

    # Fix random seeds for reproducibility
    SEED = args.seed
    torch.manual_seed(SEED)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    np.random.seed(SEED)

    experiment_log_dir = os.path.join(logs_save_dir, experiment_description, run_description,
                                      training_mode + f"_seed_{SEED}")
    os.makedirs(experiment_log_dir, exist_ok=True)

    # Logging
    log_file_name = os.path.join(experiment_log_dir, f"logs_{datetime.now().strftime('%d_%m_%Y_%H_%M_%S')}.log")
    logger = _logger(log_file_name)
    logger.debug("=" * 45)
    logger.debug(f'Dataset: {data_type}')
    logger.debug(f'Mode:    {training_mode}')
    logger.debug(f'CoFT:    {"Enabled" if args.enable_coft else "Disabled"}')
    logger.debug("=" * 45)

    try:
        # Load datasets
        data_path = os.path.join(args.data_path, data_type)
        train_dl, valid_dl, test_dl = data_generator(data_path, configs, training_mode)
        logger.debug("Data loaded ...")

        # Load Model
        model = base_Model(configs).to(device)
        temporal_contr_model = TC(configs, device).to(device)

        # CoFT: Initialize frequency branch and co-training components conditionally
        frequency_model = None
        frequency_contr_model = None
        frequency_optimizer = None
        frequency_contr_optimizer = None

        if args.enable_coft:
            frequency_model = FrequencyModel(configs).to(device)
            frequency_contr_model = FrequencyContrastive(configs, device).to(device)
            
            # Initialize optimizers for frequency components
            frequency_optimizer = torch.optim.Adam(frequency_model.parameters(), lr=configs.lr,
                                                  betas=(configs.beta1, configs.beta2), weight_decay=3e-4)
            frequency_contr_optimizer = torch.optim.Adam(frequency_contr_model.parameters(), lr=configs.lr,
                                                       betas=(configs.beta1, configs.beta2), weight_decay=3e-4)
            
            logger.debug("CoFT: Frequency branch and optimizers initialized")

        # Model loading logic based on training mode
        if "fine_tune" in training_mode or "ft_" in training_mode:
            # load saved model of this experiment
            if 'SupCon' not in training_mode:
                load_from = os.path.join(
                    os.path.join(logs_save_dir, experiment_description, run_description, f"self_supervised_seed_{SEED}",
                                 "saved_models"))
            else:
                load_from = os.path.join(
                    os.path.join(logs_save_dir, experiment_description, run_description, f"SupCon_seed_{SEED}", "saved_models"))
            chkpoint = torch.load(os.path.join(load_from, "ckp_last.pt"), map_location=device)
            pretrained_dict = chkpoint["model_state_dict"]
            model_dict = model.state_dict()
            del_list = ['logits']
            pretrained_dict_copy = pretrained_dict.copy()
            for i in pretrained_dict_copy.keys():
                for j in del_list:
                    if j in i:
                        del pretrained_dict[i]
            model_dict.update(pretrained_dict)
            model.load_state_dict(model_dict)

        if training_mode == "gen_pseudo_labels":
            ft_perc = "1p"
            load_from = os.path.join(
                os.path.join(logs_save_dir, experiment_description, run_description, f"ft_{ft_perc}_seed_{SEED}", "saved_models"))
            chkpoint = torch.load(os.path.join(load_from, "ckp_last.pt"), map_location=device)
            pretrained_dict = chkpoint["model_state_dict"]
            model.load_state_dict(pretrained_dict)
            gen_pseudo_labels(model, train_dl, device, data_path)
            logger.debug(f"✅ {mode_name} completed successfully!")
            return True

        if "train_linear" in training_mode or "tl" in training_mode:
            if 'SupCon' not in training_mode:
                load_from = os.path.join(
                    os.path.join(logs_save_dir, experiment_description, run_description, f"self_supervised_seed_{SEED}",
                                 "saved_models"))
            else:
                load_from = os.path.join(
                    os.path.join(logs_save_dir, experiment_description, run_description, f"SupCon_seed_{SEED}", "saved_models"))
            chkpoint = torch.load(os.path.join(load_from, "ckp_last.pt"), map_location=device)
            pretrained_dict = chkpoint["model_state_dict"]
            model_dict = model.state_dict()

            # 1. filter out unnecessary keys
            pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict}

            # delete these parameters (Ex: the linear layer at the end)
            del_list = ['logits']
            pretrained_dict_copy = pretrained_dict.copy()
            for i in pretrained_dict_copy.keys():
                for j in del_list:
                    if j in i:
                        del pretrained_dict[i]

            model_dict.update(pretrained_dict)
            model.load_state_dict(model_dict)
            set_requires_grad(model, pretrained_dict, requires_grad=False)  # Freeze everything except last layer.

        model_optimizer = torch.optim.Adam(model.parameters(), lr=configs.lr, betas=(configs.beta1, configs.beta2),
                                           weight_decay=3e-4)

        temporal_contr_optimizer = torch.optim.Adam(temporal_contr_model.parameters(), lr=configs.lr,
                                                    betas=(configs.beta1, configs.beta2), weight_decay=3e-4)

        if training_mode == "self_supervised" or training_mode == "SupCon":  # to do it only once
            copy_Files(os.path.join(logs_save_dir, experiment_description, run_description), data_type)

        # CUDA Optimizations
        if torch.cuda.is_available():
            torch.backends.cudnn.benchmark = True
            torch.backends.cudnn.deterministic = False
            torch.backends.cudnn.allow_tf32 = False
            torch.backends.cuda.matmul.allow_tf32 = False
            torch.cuda.empty_cache()
            print(f"CUDA optimizations enabled for {torch.cuda.get_device_name()}")

        # Trainer - Choose between original and CoFT trainer based on feature flag
        if args.enable_coft:
            CoFTTrainer(model, temporal_contr_model, frequency_model, frequency_contr_model,
                       model_optimizer, temporal_contr_optimizer, frequency_optimizer, frequency_contr_optimizer,
                       train_dl, valid_dl, test_dl, device, logger, configs, experiment_log_dir, training_mode, args.enable_coft)
        else:
            Trainer(model, temporal_contr_model, model_optimizer, temporal_contr_optimizer, train_dl, valid_dl, test_dl, device,
                    logger, configs, experiment_log_dir, training_mode)

        if training_mode != "self_supervised" and training_mode != "SupCon" and training_mode != "SupCon_pseudo":
            # Testing
            if args.enable_coft:
                outs = coft_model_evaluate(model, temporal_contr_model, frequency_model, test_dl, device, training_mode)
            else:
                outs = model_evaluate(model, temporal_contr_model, test_dl, device, training_mode)
            total_loss, total_acc, pred_labels, true_labels = outs
            _calc_metrics(pred_labels, true_labels, experiment_log_dir, args.home_path)

        elapsed = datetime.now() - overall_start_time
        logger.debug(f"✅ {mode_name} completed successfully! Total elapsed: {elapsed}")
        print(f"✅ {mode_name} completed successfully!")
        return True
        
    except Exception as e:
        print(f"❌ Error in {mode_name}: {str(e)}")
        logger.error(f"❌ Error in {mode_name}: {str(e)}")
        return False

print("✅ Main execution function loaded successfully!")


In [ ]:
# =============================================================================
# RUN THE COMPLETE TRAINING PIPELINE
# =============================================================================

def training_orchestrator(args):
    """Orchestrator function to run multiple training modes sequentially."""
    # Define the complete training pipeline sequence
    TRAINING_PIPELINE = [
        "self_supervised",
        "train_linear_1p", 
        "ft_1p",
        "gen_pseudo_labels",
        "SupCon",
        "train_linear_SupCon_1p"
    ]
    
    overall_start_time = datetime.now()
    
    print(f"\n🎯 Starting Full Training Pipeline")
    print(f"📋 Pipeline: {' → '.join(TRAINING_PIPELINE)}")
    print(f"🗂️ Dataset: {args.selected_dataset}")
    print(f"🔄 CoFT: {'Enabled' if args.enable_coft else 'Disabled'}")
    print(f"⏰ Start Time: {overall_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    successful_modes = []
    failed_modes = []
    
    for i, mode in enumerate(TRAINING_PIPELINE, 1):
        print(f"\n📍 Step {i}/{len(TRAINING_PIPELINE)}: {mode}")
        
        # Execute the training mode
        success = execute_training_mode(args, mode, overall_start_time)
        
        if success:
            successful_modes.append(mode)
            print(f"✅ Step {i} completed: {mode}")
        else:
            failed_modes.append(mode)
            print(f"❌ Step {i} failed: {mode}")
            print(f"🛑 Stopping pipeline due to failure in {mode}")
            break
    
    # Final summary
    total_time = datetime.now() - overall_start_time
    print(f"\n{'='*80}")
    print(f"🏁 TRAINING PIPELINE SUMMARY")
    print(f"{'='*80}")
    print(f"⏱️ Total Time: {total_time}")
    print(f"✅ Successful: {len(successful_modes)}/{len(TRAINING_PIPELINE)} modes")
    if successful_modes:
        print(f"   {' → '.join(successful_modes)}")
    if failed_modes:
        print(f"❌ Failed: {failed_modes}")
    print(f"{'='*80}")
    
    if len(successful_modes) == len(TRAINING_PIPELINE):
        print("🎉 FULL PIPELINE COMPLETED SUCCESSFULLY!")
        return True
    else:
        print("⚠️ PIPELINE INCOMPLETE - CHECK FAILED MODES")
        return False

# =============================================================================
# EXECUTE THE TRAINING
# =============================================================================

# Create arguments object from configuration
class Args:
    def __init__(self, config):
        self.experiment_description = config.experiment_description
        self.run_description = config.run_description
        self.seed = config.seed
        self.training_mode = config.training_mode
        self.selected_dataset = config.selected_dataset
        self.data_path = config.data_path
        self.logs_save_dir = config.logs_save_dir
        self.device = config.device
        self.home_path = config.home_path
        self.enable_coft = config.enable_coft

# Convert configuration to arguments
args = Args(config)

print("\n" + "="*80)
print("🚀 STARTING COFT EXPERIMENT")
print("="*80)
print(f"📋 Configuration Summary:")
print(f"   Experiment: {args.experiment_description}")
print(f"   Run: {args.run_description}")
print(f"   Dataset: {args.selected_dataset}")
print(f"   CoFT: {'Enabled' if args.enable_coft else 'Disabled'}")
print(f"   Device: {args.device}")
print(f"   Training Mode: {args.training_mode}")

start_time = datetime.now()

# Check if orchestrator mode is requested
if args.training_mode == "full_run":
    print("🚀 FULL TRAINING PIPELINE MODE ACTIVATED")
    success = training_orchestrator(args)
    if success:
        print("\n🎉 EXPERIMENT COMPLETED SUCCESSFULLY!")
    else:
        print("\n⚠️ EXPERIMENT FAILED - CHECK LOGS FOR DETAILS")
else:
    # Single mode execution
    print(f"🎯 SINGLE MODE EXECUTION: {args.training_mode}")
    success = execute_training_mode(args, args.training_mode, start_time)
    if success:
        print("\n✅ SINGLE MODE COMPLETED SUCCESSFULLY!")
    else:
        print("\n❌ SINGLE MODE FAILED - CHECK LOGS FOR DETAILS")

print("\\n" + "="*80)
print("🏁 EXPERIMENT FINISHED")
print("="*80)
